In [1]:
import pandas as pd
import xlrd
import openpyxl
from pandas import ExcelWriter
from datetime import datetime
import os
from shutil import copyfile
from openpyxl.utils.cell import get_column_letter
import urllib.request, json
import geopoll_functions
import geopoll_modified
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
import re

## Possible changes ##

In [1]:
import os
import shutil
from datetime import datetime

def refresh_local_template_folder_if_changed(
    synced_sharepoint_dir: str,
    local_templates_dir: str,
    archive_folder_name: str = "Archive",
    template_prefixes=("household_questionnaire_",),
    extensions=(".xlsx",),
    compare_mode: str = "mtime_size",   # "mtime_size" (fast) or "hash" (accurate)
    dry_run: bool = False
):
    """
    Only archives + copies if templates differ between:
      - synced_sharepoint_dir (source, read-only)
      - local_templates_dir (destination)

    If unchanged: prints a message and does nothing.
    If changed: creates Archive/<timestamp>/, moves existing local templates there, copies synced ones into local folder.
    """

    def is_template_file(filename: str) -> bool:
        fn = filename.lower()
        if not fn.endswith(tuple([e.lower() for e in extensions])):
            return False
        return any(fn.startswith(p.lower()) for p in template_prefixes)

    # --- collect source and dest files (by name) ---
    sp_files = sorted([f for f in os.listdir(synced_sharepoint_dir)
                       if os.path.isfile(os.path.join(synced_sharepoint_dir, f)) and is_template_file(f)])

    local_files = sorted([f for f in os.listdir(local_templates_dir)
                          if os.path.isfile(os.path.join(local_templates_dir, f)) and is_template_file(f)])

    sp_set = set(sp_files)
    local_set = set(local_files)

    # If file lists differ (added/removed/renamed) => definitely changed
    if sp_set != local_set:
        changed = True
        reason = "file list differs (added/removed templates)"
    else:
        # Same names: compare metadata or hashes
        changed = False
        reason = "all templates identical"

        if compare_mode == "mtime_size":
            for f in sp_files:
                sp_path = os.path.join(synced_sharepoint_dir, f)
                loc_path = os.path.join(local_templates_dir, f)

                sp_stat = os.stat(sp_path)
                loc_stat = os.stat(loc_path)

                # Compare size + modified time (rounded to seconds)
                if sp_stat.st_size != loc_stat.st_size or int(sp_stat.st_mtime) != int(loc_stat.st_mtime):
                    changed = True
                    reason = f"file changed: {f}"
                    break

        elif compare_mode == "hash":
            import hashlib

            def file_hash(path, chunk=1024 * 1024):
                h = hashlib.sha256()
                with open(path, "rb") as fp:
                    while True:
                        b = fp.read(chunk)
                        if not b:
                            break
                        h.update(b)
                return h.hexdigest()

            for f in sp_files:
                sp_path = os.path.join(synced_sharepoint_dir, f)
                loc_path = os.path.join(local_templates_dir, f)
                if file_hash(sp_path) != file_hash(loc_path):
                    changed = True
                    reason = f"file content changed: {f}"
                    break
        else:
            raise ValueError("compare_mode must be 'mtime_size' or 'hash'")

    # --- If no change, exit gracefully ---
    if not changed:
        print("✅ Templates are already up to date. No files moved to Archive.")
        return None

    # --- If changed, archive + copy ---
    ts = datetime.now().strftime("%Y%m%d%H%M%S")
    archive_root = os.path.join(local_templates_dir, archive_folder_name)
    archive_dir = os.path.join(archive_root, ts)

    print(f"🔄 Templates update detected ({reason}).")
    print(f"📦 Archiving current templates to: {archive_dir}")

    if not dry_run:
        os.makedirs(archive_dir, exist_ok=True)

        # Move existing local templates (only those matching the filter)
        for f in local_files:
            shutil.move(os.path.join(local_templates_dir, f), os.path.join(archive_dir, f))

        # Copy synced templates into local folder
        for f in sp_files:
            shutil.copy2(os.path.join(synced_sharepoint_dir, f), os.path.join(local_templates_dir, f))

    else:
        print("[DRY RUN] Would move local templates and copy new ones.")

    print("✅ Local templates refreshed.")
    return archive_dir

In [ ]:
template_questionnaire_synced_sharepoint = r"C:\Users\edoar\Food and Agriculture Organization\OER - 01. DIEM Monitoring\03. Toolbox\02. Questionnaires"  # your synced folder
template_questionnaire_location = r"C:\Users\edoar\WORK\FAO\repo\questionnaire_validation\Questionnaire Validation\Questionnaires\Questionnaire Templates"

refresh_local_template_folder_if_changed(
    synced_sharepoint_dir=template_questionnaire_synced_sharepoint,
    local_templates_dir=template_questionnaire_location,
    compare_mode="mtime_size"      
)

✅ Templates are already up to date. No files moved to Archive.


In [ ]:
#to use to possibly detect the template language from the filename, to then find the latest template for that language in the templates folder
def detect_last_template(file_name: str, template_folder: str, enumerator: str = "geopoll") -> str:
    """
    Returns the filename (NOT full path) of the latest template matching:
      household_questionnaire_<enumerator>_<LANG>_template_<YYYYMMDD>_ISO3*.xlsx

    - LANG is detected from file_name using detect_language()
    - Chooses the latest by the numeric template version in the filename
    - Falls back to modified time if version cannot be parsed
    """

    language = geopoll_functions.detect_language(file_name).upper()  # EN/FR/ES/AR/PT
    if language not in {"EN", "FR", "ES", "AR", "PT"}:
        raise ValueError(f"Language not recognized from filename: {file_name}")

    # Match both geopoll patterns if you ever want (right now enumerator=geopoll)
    # Examples:
    # household_questionnaire_geopoll_EN_template_20250708_ISO3.xlsx
    pattern = re.compile(
        rf"^household_questionnaire_{re.escape(enumerator)}_{language}_template_(\d{{8}})_ISO3.*\.xlsx$",
        re.IGNORECASE
    )

    candidates = []
    for f in os.listdir(template_folder):
        m = pattern.match(f)
        if m:
            ver = m.group(1)  # YYYYMMDD
            try:
                ver_int = int(ver)
            except:
                ver_int = None

            full = os.path.join(template_folder, f)
            mtime = os.path.getmtime(full)
            candidates.append((ver_int, mtime, f))

    if not candidates:
        raise FileNotFoundError(
            f"No templates found for {enumerator} {language} in: {template_folder}"
        )

    # Prefer the highest version number; if missing, use mtime
    # Sort by: version (None last), then mtime
    candidates.sort(key=lambda x: ((x[0] is None), -(x[0] or 0), -x[1]))

    return candidates[0][2]

In [19]:
from openpyxl import load_workbook

def check_questionnaire_workbook(file_path: str, required_sheets=None, stop_on_fail=True):
    if required_sheets is None:
        required_sheets = ["survey", "Crop list", "Additional information"]

    issues = []
    info = []

    try:
        wb = load_workbook(file_path, data_only=True)
    except Exception as e:
        # This often happens if the file is password-encrypted (cannot be opened)
        msg = f"❌ Cannot open Excel file. It may be password-protected/encrypted or corrupted.\n   {e}"
        if stop_on_fail:
            raise RuntimeError(msg)
        print(msg)
        return {"ok": False, "issues": [msg], "info": []}

    # ---- 1) Check required sheets exist (case-insensitive match) ----
    existing = wb.sheetnames
    existing_lower = {s.lower(): s for s in existing}  # map lowercase -> actual name

    missing = []
    for rs in required_sheets:
        if rs.lower() not in existing_lower:
            missing.append(rs)

    if missing:
        issues.append(f"❌ Missing required sheet(s): {missing}")
    else:
        info.append("✅ All required sheets are present.")

    # ---- 2) Check workbook protection (structure/windows) ----
    # Not always set; if None, treat as not protected at workbook-level
    wb_prot = wb.security
    wb_structure = bool(getattr(wb_prot, "lockStructure", False))
    wb_windows = bool(getattr(wb_prot, "lockWindows", False))

    # ---- 3) Check per-sheet protection ----
    protected_sheets = []
    for s in existing:
        ws = wb[s]
        # ws.protection.sheet == True means sheet protection enabled (may or may not have password)
        if getattr(ws.protection, "sheet", False):
            protected_sheets.append(s)

    if wb_structure or wb_windows or protected_sheets:
        info.append(
            "🔒 Protection detected: "
            f"workbook(lockStructure={wb_structure}, lockWindows={wb_windows}), "
            f"protected sheets={protected_sheets if protected_sheets else 'none'}"
        )
    else:
        info.append("🔓 No workbook/sheet protection detected (structure/windows/sheet flags are off).")

    # ---- decide OK ----
    ok = (len(issues) == 0)

    if not ok and stop_on_fail:
        raise RuntimeError("\n".join(issues))

    # Print a short summary (nice for notebook logs)
    for line in info:
        print(line)
    for line in issues:
        print(line)

    return {"ok": ok, "issues": issues, "info": info}

In [23]:
check_questionnaire_workbook(
    file_path=country_questionnaire_file,
    required_sheets=["survey", "Crop list", "Additional information"],
    stop_on_fail=True
)

RuntimeError: ❌ Missing required sheet(s): ['Additional information']

## Previous version of the code ##

In [ ]:
# =============================================================================
# 🛠 USER CONFIGURATION: Define metadata and file locations
# =============================================================================

# --- Basic Survey Metadata ---
template_version = "20250708"
iso3 = "MMR"
round_number = "13"
admin_level = "Admin 2"
user_name = "Edoardo"

# --- Questionnaire File Names ---
country_questionnaire_file = r"MMR_R13 household_questionnaire_t1_geopoll_EN_template_20250708_ISO3.xlsx"
previousround_questionnaire = "no"       # "yes" if current form is derived from a validated previous version
previousround_questionnaire_file = r"validated_questionnaire_geopoll_en_MMR_20240305122409.xlsx"


# --- File Path Configuration ---
template_questionnaire_location = r'C:\Users\edoar\WORK\FAO\repo\questionnaire_validation\Questionnaire Validation\Questionnaires\Questionnaire Templates'
previousround_questionnaire_location = r'C:\Users\edoar\WORK\FAO\repo\questionnaire_validation\Questionnaire Validation\Questionnaires\Validated Questionnaires\MMR\R9'
country_questionnaires_location = r'C:\Users\edoar\WORK\FAO\repo\questionnaire_validation\Questionnaire Validation\Questionnaires\Questionnaire Country'
output_locations = r"C:\Users\edoar\WORK\FAO\temp\questionnaireValidationTest"
# =============================================================================
# END OF USER INPUTS
# =============================================================================

# =============================================================================
# Questionnaire Metadata Initialization
# =============================================================================
validation_number = 1
enumerator = geopoll_functions.detect_enumerator(country_questionnaire_file)
lang = geopoll_functions.detect_language(country_questionnaire_file)
startTime = datetime.now()
now = startTime.strftime('%Y%m%d%H%M%S')                  # For filenames
formatted_now = startTime.strftime("%d/%m/%Y, %H:%M:%S")   # For logging

# Initialize execution log
execution_messages = "\n📋 Execution Summary:"
execution_messages += f"\n- Country ISO3: {iso3}"
execution_messages += f"\n- Round #: {round_number}"
execution_messages += f"\n- Questionnaire Language: {lang}"
execution_messages += f"\n- Enumerator Type: {enumerator}"
execution_messages += f"\n- Current Questionnaire File: {country_questionnaire_file}"

country_questionnaire_file = os.path.join(country_questionnaires_location, country_questionnaire_file)

template_questionnaire_file = detect_last_template(
    file_name=country_questionnaire_file,
    template_folder=template_questionnaire_location,
    enumerator="geopoll"   # or enumerator variable if you want
)
template_questionnaire_file = os.path.join(template_questionnaire_location, template_questionnaire_file)

##OLD version
#template_questionnaire_file = geopoll_functions.detect_template(template_version,country_questionnaire_file)
#template_questionnaire_file = os.path.join(template_questionnaire_location,template_questionnaire_file)



execution_messages += f"\n- Template Version Used: {template_version}"

if previousround_questionnaire.lower() == "yes":
    previousround_questionnaire_file = os.path.join(previousround_questionnaire_location, previousround_questionnaire_file)
    execution_messages += f"\n- Previous Round File: {previousround_questionnaire_file}"

execution_messages += f"\n- Validation Number: {validation_number}"
execution_messages += f"\n- Analyst: {user_name}"
execution_messages += f"\n- Validated On: {formatted_now}"

print(execution_messages)


📋 Execution Summary:
- Country ISO3: MMR
- Round #: 13
- Questionnaire Language: en
- Enumerator Type: geopoll
- Current Questionnaire File: MMR_R13 household_questionnaire_t1_geopoll_EN_template_20250708_ISO3.xlsx
- Template Version Used: 20250708
- Validation Number: 1
- Analyst: Edoardo
- Validated On: 26/02/2026, 14:36:06


In [ ]:
print("Checking the conditionality of all questions...")

questions_conditionality = os.path.join(
    output_locations,
    f"conditionality_questionnaire_{enumerator}_{lang}_{iso3}_{now}.xlsx"
)

conditionality_result_brief, conditionality_result_details = geopoll_functions.check_all_questions_unified(
    questionnaire_file=country_questionnaire_file,
    template_file=template_questionnaire_file,
    result_file=questions_conditionality,
    previousround_questionnaire=previousround_questionnaire,
    previous_questionnaire_file=previousround_questionnaire_file
)

print("\n--- Conditionality Check Summary ---")
print(conditionality_result_brief)
print("File: " + f"conditionality_questionnaire_{enumerator}_{lang}_{iso3}_{now}.xlsx")

Checking the conditionality of all questions...

--- Conditionality Check Summary ---
❌ Detected 1281 mismatches across 19 fields.
File: conditionality_questionnaire_geopoll_en_MMR_20260226114232.xlsx


In [ ]:
# --- TEMPLATE QUESTIONNAIRE ---

print("Reading TEMPLATE questionnaire and creating recap Excel file")
questionnaire_file = template_questionnaire_file
temp_path = output_locations
coded_values_file = os.path.join(temp_path, f"coded_values_{now}_template.xlsx")
field_names_list = []
max_counter = 6000
languages = geopoll_functions.read_questionnaire(questionnaire_file, coded_values_file)


# --- CURRENT QUESTIONNAIRE ---

print("Reading CURRENT questionnaire and creating recap Excel file")
country_coded_values_file = os.path.join(temp_path, f"coded_values_{now}_country.xlsx")
field_names_list = []
max_counter = 6000
languages = geopoll_functions.read_questionnaire(country_questionnaire_file, country_coded_values_file)


# --- PREVIOUS ROUND QUESTIONNAIRE (optional) ---

if previousround_questionnaire.strip().lower() == "yes":
    print("Reading PREVIOUS ROUND questionnaire and creating recap Excel file")
    
    previous_questionnaire_file = previousround_questionnaire_file

    previous_coded_values_file = os.path.join(temp_path, f"coded_values_{now}_previous.xlsx")
    field_names_list = []
    max_counter = 6000
    languages = geopoll_functions.read_questionnaire(previous_questionnaire_file, previous_coded_values_file)

Reading TEMPLATE questionnaire and creating recap Excel file
Version in English only
Reading CURRENT questionnaire and creating recap Excel file
Version in English only


In [ ]:
print("COUNTING NUMBER OF QUESTIONS")

T_file = questionnaire_file  # Template
C_file = country_questionnaire_file  # Current
P_file = previousround_questionnaire_file if previousround_questionnaire.strip().lower() == "yes" else None

# Count Q Name-based
T_qname_list, T_qname_count = geopoll_functions.count_number_of_questions_qname(T_file)
C_qname_list, C_qname_count = geopoll_functions.count_number_of_questions_qname(C_file)
if P_file:
    P_qname_list, P_qname_count = geopoll_functions.count_number_of_questions_qname(P_file)

# Count Suggested Q Name-based
T_sqname_list, T_sqname_count = geopoll_functions.count_number_of_questions_sqname(T_file)
C_sqname_list, C_sqname_count = geopoll_functions.count_number_of_questions_sqname(C_file)
if P_file:
    P_sqname_list, P_sqname_count = geopoll_functions.count_number_of_questions_sqname(P_file)

messages = [
    "QName Field:",
    f"Template Questionnaire: {T_qname_count}",
    f"Current Questionnaire: {C_qname_count}",
]
if P_file:
    messages.append(f"Previous Questionnaire: {P_qname_count}")

messages += [
    "\nSuggestedQName Field:",
    f"Template Questionnaire: {T_sqname_count}",
    f"Current Questionnaire: {C_sqname_count}",
]
if P_file:
    messages.append(f"Previous Questionnaire: {P_sqname_count}")

for msg in messages:
    print(msg)
execution_messages += "\n\n" + "\n".join(messages)

COUNTING NUMBER OF QUESTIONS
QName Field:
Template Questionnaire: 180
Current Questionnaire: 204

SuggestedQName Field:
Template Questionnaire: 178
Current Questionnaire: 182


In [ ]:
print("CHECKING IF ALL MANDATORY QUESTIONS ARE INCLUDED IN THE CURRENT QUESTIONNAIRE")

country_questions = pd.read_excel(open(country_questionnaire_file, 'rb'), sheet_name='survey', skiprows=2)
C_qname_list_questions = country_questions["Q Name"].dropna().astype(str).str.strip().unique().tolist()

if previousround_questionnaire.strip().lower() == "yes":
    previous_questions = pd.read_excel(open(previousround_questionnaire_file, 'rb'), sheet_name='survey', skiprows=2)
    P_qname_list_questions = previous_questions["Q Name"].dropna().astype(str).str.strip().unique().tolist()
else:
    P_qname_list_questions = []

template_questions = pd.read_excel(open(questionnaire_file, 'rb'), sheet_name='survey', skiprows=2)

list_mandatory_questions = template_questions.loc[
    (~template_questions["Q Name"].astype(str).str.contains("hh_wealth_", na=False)) &
    (template_questions["Mandatory"].astype(str).str.strip().str.lower() == "yes"),
    "Q Name"
].dropna().astype(str).str.strip().tolist()

message9 = f"\nFound {len(list_mandatory_questions)} mandatory questions in the template questionnaire"
print(message9)

# ---------------------------------------
# Admin2/Admin3 variant detection patterns
# ---------------------------------------
admin2_pattern = re.compile(r"(?:^|_)adm(?:in)?_?2(?:_|$)|(?:^|_)admin2(?:_|$)|(?:^|_)hh_admin2(?:_|$)|(?:^|_)adm2(?:_|$)", re.IGNORECASE)
admin3_pattern = re.compile(r"(?:^|_)adm(?:in)?_?3(?:_|$)|(?:^|_)admin3(?:_|$)|(?:^|_)hh_admin3(?:_|$)|(?:^|_)adm3(?:_|$)", re.IGNORECASE)

admin2_exists_country  = any(admin2_pattern.search(q) for q in C_qname_list_questions)
admin2_exists_previous = any(admin2_pattern.search(q) for q in P_qname_list_questions)

admin3_exists_country  = any(admin3_pattern.search(q) for q in C_qname_list_questions)
admin3_exists_previous = any(admin3_pattern.search(q) for q in P_qname_list_questions)

# -----------------------------
# Template → Country comparison
# -----------------------------
missing_vs_country = list(set(list_mandatory_questions) - set(C_qname_list_questions))

# ------------------------------------
# Template → Previous Round comparison
# ------------------------------------
missing_vs_previous = list(set(list_mandatory_questions) - set(P_qname_list_questions)) if P_qname_list_questions else []

# ----------------------------------------
# Previous Round → Current (dropped mandatory)
# ----------------------------------------
mandatory_previous_questions = [q for q in P_qname_list_questions if q in list_mandatory_questions]
missing_from_previous_to_country = list(set(mandatory_previous_questions) - set(C_qname_list_questions)) if P_qname_list_questions else []

# -------------------------------------------------------
# Apply tolerance: if admin2/admin3 exists under ANY name,
# don't flag admin2/admin3 mandatory items as missing.
# -------------------------------------------------------
def _filter_admin_variants(missing_list, admin2_exists, admin3_exists):
    filtered = []
    for q in missing_list:
        if admin2_exists and admin2_pattern.search(q):
            continue
        if admin3_exists and admin3_pattern.search(q):
            continue
        filtered.append(q)
    return filtered

missing_vs_country = _filter_admin_variants(missing_vs_country, admin2_exists_country, admin3_exists_country)
missing_vs_previous = _filter_admin_variants(missing_vs_previous, admin2_exists_previous, admin3_exists_previous)
missing_from_previous_to_country = _filter_admin_variants(missing_from_previous_to_country, admin2_exists_country, admin3_exists_country)

status_country  = "✅ All Present" if not missing_vs_country else "❌ Not All Found"
status_previous = "✅ All Present" if not missing_vs_previous else "❌ Not All Found"
status_dropped  = "✅ No Dropped Mandatory" if not missing_from_previous_to_country else "❌ Questions Dropped"


message_summary = "\nMandatory Questions Check\n"
message_summary += "| Compared To              | Status                  | Missing Count | Action                              |\n"
message_summary += "|--------------------------|--------------------------|----------------|--------------------------------------|\n"
message_summary += f"| Current vs. Prev. Round | {status_dropped:<24} | {len(missing_from_previous_to_country):<14} | {'Review dropped mandatory questions' if missing_from_previous_to_country else 'No action needed'} |\n"
message_summary += f"| Current vs. Template    | {status_country:<24} | {len(missing_vs_country):<14} | {'Review missing questions' if missing_vs_country else 'No action needed'} |\n"
if previousround_questionnaire.strip().lower() == "yes":
    message_summary += f"| Template vs Prev. Round | {status_previous:<24} | {len(missing_vs_previous):<14} | {'Check previous template completeness' if missing_vs_previous else 'No action needed'} |\n"


details = "\nMissing Mandatory Questions:\n"
if missing_from_previous_to_country:
    details += f"\n\nDropped from Previous Round:\n- " + "\n- ".join(sorted(missing_from_previous_to_country))
if missing_vs_country:
    details += f"\nFrom Template (vs. Current):\n- " + "\n- ".join(sorted(missing_vs_country))
if missing_vs_previous:
    details += f"\n\nFrom Template (vs. Previous Round):\n- " + "\n- ".join(sorted(missing_vs_previous))

if not any([missing_vs_country, missing_vs_previous, missing_from_previous_to_country]):
    details += "None — all mandatory questions are present."

print(message_summary)
print(details)
execution_messages += message9 + "\n" + message_summary + "\n" + details + "\n"

CHECKING IF ALL MANDATORY QUESTIONS ARE INCLUDED IN THE CURRENT QUESTIONNAIRE

Found 81 mandatory questions in the template questionnaire

Mandatory Questions Check
| Compared To              | Status                  | Missing Count | Action                              |
|--------------------------|--------------------------|----------------|--------------------------------------|
| Current vs. Prev. Round | ✅ No Dropped Mandatory   | 0              | No action needed |
| Current vs. Template    | ❌ Not All Found          | 7              | Review missing questions |


Missing Mandatory Questions:

From Template (vs. Current):
- callback
- hdds
- hdds_confirmation
- hh_admin2_1
- need_2
- resp_firstname
- resp_lastname


In [ ]:
print("\n\nCHECKING OPTIONAL QUESTIONS ADDED IN THE CURRENT QUESTIONNAIRE")

template_questions = pd.read_excel(open(questionnaire_file, 'rb'), sheet_name='survey', skiprows=2)
template_questions.columns = template_questions.columns.str.strip()
T_qname_list_questions = template_questions["Q Name"].dropna().str.strip().unique().tolist()

optional_in_country = [q for q in C_qname_list_questions if q.startswith("o_")]
optional_in_template = [q for q in T_qname_list_questions if q.startswith("o_")]

added_optional_questions = sorted(set(optional_in_country) - set(optional_in_template))

if previousround_questionnaire == "yes":
    previous_questions = pd.read_excel(open(previousround_questionnaire_file, 'rb'), sheet_name='survey', skiprows=2)
    previous_questions.columns = previous_questions.columns.str.strip()
    P_qname_list_questions = previous_questions["Q Name"].dropna().str.strip().unique().tolist()
    optional_in_previous = [q for q in P_qname_list_questions if q.startswith("o_")]

    dropped_optional_questions = sorted(set(optional_in_previous) - set(optional_in_country))

    newly_added_optional = sorted(set(optional_in_country) - set(optional_in_template) - set(optional_in_previous))
else:
    optional_in_previous = []
    dropped_optional_questions = []
    newly_added_optional = added_optional_questions

aligned_optional_questions = sorted(set(optional_in_country) & (set(optional_in_template) | set(optional_in_previous)))

if aligned_optional_questions:
    print(f"\n{len(aligned_optional_questions)} optional questions match those in the previous round:\n")
    for q in aligned_optional_questions:
        print(f"- {q}")
    execution_messages += f"\nAligned optional questions (matching previous):\n" + "\n".join(f"- {q}" for q in aligned_optional_questions) + "\n"

if newly_added_optional:
    print(f"\n➕ {len(newly_added_optional)} newly added optional questions (not in previous):")
    for q in newly_added_optional:
        print(f"- {q}")
    execution_messages += f"\n➕ Newly added optional questions:\n" + "\n".join(f"- {q}" for q in newly_added_optional) + "\n"

if dropped_optional_questions:
    print(f"\n➖ {len(dropped_optional_questions)} optional questions were dropped (present in previous round but not in current):")
    for q in dropped_optional_questions:
        print(f"- {q}")
    execution_messages += f"\n➖ Dropped optional questions:\n" + "\n".join(f"- {q}" for q in dropped_optional_questions) + "\n"

if not newly_added_optional and not dropped_optional_questions:
    print("✅ No optional questions added or dropped.")
    execution_messages += "\n✅ No optional questions added or dropped.\n"



CHECKING OPTIONAL QUESTIONS ADDED IN THE CURRENT QUESTIONNAIRE

➕ 50 newly added optional questions (not in previous):
- o_AccessMarket_why
- o_ExpenseFood30d
- o_HHDisability
- o_HHExpTotal30d
- o_HHSize04
- o_HHSize16_59
- o_HHSize5_15
- o_HHSize60
- o_Income_total_amount
- o_Income_total_amount1
- o_Income_total_amount_conf
- o_Income_total_comp
- o_crp_returnee_plotaccess
- o_diff_crp_input
- o_diff_crp_output
- o_diff_ls_input
- o_diff_ls_output
- o_diff_marketfunc1
- o_hh_admin3_1
- o_hh_admin3_11
- o_hh_admin3_12
- o_hh_admin3_13
- o_hh_admin3_14
- o_hh_admin3_15
- o_hh_admin3_2
- o_hh_admin3_3
- o_hh_admin3_4
- o_hh_admin3_5
- o_hh_admin3_6
- o_hh_admin3_7
- o_hh_admin3_8
- o_hh_admin3_9
- o_hh_arrivaltime
- o_hh_displacementfreq
- o_hh_residencecomp
- o_hh_shelter
- o_hh_site
- o_hh_wealth_house
- o_income_main_selfemployed
- o_income_sec_selfemployed
- o_income_third_selfemployed
- o_incomechangereduced
- o_ls_feed_freedist
- o_meals
- o_wage_main_amount
- o_wage_main_amoun

In [ ]:
print("DETECTING DIFFERENCES IN FIELD: Suggested QName")

# -- Load and clean column names --
template_questions.columns = template_questions.columns.str.strip()
country_questions.columns = country_questions.columns.str.strip()
if previousround_questionnaire.strip().lower() == "yes":
    previous_questions.columns = previous_questions.columns.str.strip()

# -- Helper to find closest match for 'Suggested QName' column --
def find_suggested_qname_column(df, label=""):
    for col in df.columns:
        if "suggested" in col.lower() and "qname" in col.lower():
            return col
    raise KeyError(f"❌ 'Suggested QName' column not found in {label} file. Available columns: {df.columns.tolist()}")

# -- Identify column names safely --
T_qname_col = find_suggested_qname_column(template_questions, "template")
C_qname_col = find_suggested_qname_column(country_questions, "current")
if previousround_questionnaire.strip().lower() == "yes":
    P_qname_col = find_suggested_qname_column(previous_questions, "previous round")

# -- Prepare Suggested QName lists --
T_sqname_list_questions = template_questions[T_qname_col].dropna().astype(str).str.strip().unique().tolist()
C_sqname_list_questions = country_questions[C_qname_col].dropna().astype(str).str.strip().unique().tolist()
if previousround_questionnaire.strip().lower() == "yes":
    P_sqname_list_questions = previous_questions[P_qname_col].dropna().astype(str).str.strip().unique().tolist()

# -- Difference calculation --
raw_template_missing = set(T_sqname_list_questions) - set(C_sqname_list_questions)
raw_country_only = set(C_sqname_list_questions) - set(T_sqname_list_questions)

# Deduplicate: remove overlapping entries (e.g. if fcs_intro in both sets, keep it in one only)
clean_template_missing = sorted(raw_template_missing - raw_country_only)
clean_country_only = sorted(raw_country_only - raw_template_missing)

# Previous vs Current difference (only used for printout)
if previousround_questionnaire.strip().lower() == "yes":
    unique_previousround_questionnaire = sorted(set(P_sqname_list_questions) - set(C_sqname_list_questions))

# -- Results summary table --
summary_table = "\nSuggestedQName Differences Summary\n"
summary_table += "| Compared To       | Status             | Difference Count | Severity | Action                         |\n"
summary_table += "|-------------------|--------------------|------------------|----------|--------------------------------|\n"

template_diff_count = len(clean_template_missing)
template_status = "✅ No Differences" if template_diff_count == 0 else "❌ Differences Found"
template_severity = "Low" if template_diff_count == 0 else "Medium"
template_action = "No action needed" if template_diff_count == 0 else "Review added/missing questions"
summary_table += f"| Template           | {template_status:<19} | {template_diff_count:<16} | {template_severity:<8} | {template_action:<30} |\n"

if previousround_questionnaire.strip().lower() == "yes":
    previous_diff_count = len(unique_previousround_questionnaire)
    previous_status = "✅ No Differences" if previous_diff_count == 0 else "❌ Differences Found"
    previous_severity = "Low" if previous_diff_count == 0 else "Medium"
    previous_action = "No action needed" if previous_diff_count == 0 else "Check removed questions from new round"
    summary_table += f"| Previous Round     | {previous_status:<19} | {previous_diff_count:<16} | {previous_severity:<8} | {previous_action:<30} |\n"

country_diff_count = len(clean_country_only)
country_status = "✅ All Published" if country_diff_count == 0 else "⚠️ Not Published"
country_action = "No action needed" if country_diff_count == 0 else "Validate if should keep"
summary_table += f"| Current Quest. Only | {country_status:<19} | {country_diff_count:<16} | Low      | {country_action:<30} |\n"

# -- Append and print summary --
execution_messages += summary_table
print(summary_table)

# -- Optional visual diff (if applicable) --
if previousround_questionnaire.strip().lower() == "yes":
    geopoll_functions.highlight_differences_in_qname(country_questionnaire_file, previousround_questionnaire_file)

# -- Detailed Differences (deduplicated printouts) --
if previousround_questionnaire.strip().lower() == "yes" and (previous_diff_count > 0):
    print("\nSuggested QName in Previous Round but Missing in Current Round:")
    execution_messages += "\n\nSuggested QName in Previous Round but Missing in Current Round:\n"
    for q in unique_previousround_questionnaire:
        line = f"- {q}\n"
        print(line.strip())
        execution_messages += line

if template_diff_count > 0:
    print("\nSuggested QName in Template but Missing in Current Questionnaire:")
    execution_messages += "\n\nSuggested QName in Template but Missing in Current Questionnaire:\n"
    for q in clean_template_missing:
        line = f"- {q}\n"
        print(line.strip())
        execution_messages += line

if country_diff_count > 0:
    print("\nSuggested QName in Current Questionnaire not in Template:")
    execution_messages += "\n\nSuggested QName in Current Questionnaire not in Template:\n"
    for q in clean_country_only:
        line = f"- {q}\n"
        print(line.strip())
        execution_messages += line

DETECTING DIFFERENCES IN FIELD: Suggested QName

SuggestedQName Differences Summary
| Compared To       | Status             | Difference Count | Severity | Action                         |
|-------------------|--------------------|------------------|----------|--------------------------------|
| Template           | ❌ Differences Found | 73               | Medium   | Review added/missing questions |
| Current Quest. Only | ⚠️ Not Published    | 64               | Low      | Validate if should keep        |


Suggested QName in Template but Missing in Current Questionnaire:
- assistance_provider

1) assistance_fao
2) assistance_wfp
3) assistance_otherun
4) assistance_gov
5) assistance_ngo
6) assistance_dk
7) assistance_ref
- callback
- contact_2
- contact_2_name
- contact_3
- covid_

1) covid_goodstransp
2) covid_marketclosed
3) covid_borderclosed
4) covid_stayhome
5) covid_gatherings
6) covid_processclosed
7) covid_curfew
8) covid_school
9) covid_other
10) covid_none
11) covid_dk
12) 

In [ ]:
print("DETECTING DIFFERENCES IN FIELD: Q Name")

# -- Clean and ensure column names --
template_questions.columns = template_questions.columns.str.strip()
country_questions.columns = country_questions.columns.str.strip()
if previousround_questionnaire.strip().lower() == "yes":
    previous_questions.columns = previous_questions.columns.str.strip()

# -- Identify correct Q Name column --
def find_qname_column(df, label=""):
    for col in df.columns:
        if "q name" in col.lower() or col.strip().lower() == "qname":
            return col
    raise KeyError(f"❌ 'Q Name' column not found in {label} file. Available columns: {df.columns.tolist()}")

T_qname_col = find_qname_column(template_questions, "template")
C_qname_col = find_qname_column(country_questions, "current")
if previousround_questionnaire.strip().lower() == "yes":
    P_qname_col = find_qname_column(previous_questions, "previous round")

# -- Create lists --
T_qname_list_questions = template_questions[T_qname_col].dropna().astype(str).str.strip().unique().tolist()
C_qname_list_questions = country_questions[C_qname_col].dropna().astype(str).str.strip().unique().tolist()
if previousround_questionnaire.strip().lower() == "yes":
    P_qname_list_questions = previous_questions[P_qname_col].dropna().astype(str).str.strip().unique().tolist()

# -- Differences --
unique_country_questionnaire_qname = list(set(C_qname_list_questions) - set(T_qname_list_questions))
unique_template_questionnaire_qname = list(set(T_qname_list_questions) - set(C_qname_list_questions))
if previousround_questionnaire.strip().lower() == "yes":
    unique_previousround_questionnaire_qname = list(set(P_qname_list_questions) - set(C_qname_list_questions))

# -- Summary Table --
qname_summary = "\nQ Name Differences Summary\n"
qname_summary += "| Compared To       | Status             | Difference Count | Severity | Action                         |\n"
qname_summary += "|-------------------|---------------------|------------------|----------|--------------------------------|\n"

qname_template_diff_count = len(unique_template_questionnaire_qname)
qname_template_status = "✅ No Differences" if qname_template_diff_count == 0 else "❌ Differences Found"
qname_summary += f"| Template           | {qname_template_status:<19} | {qname_template_diff_count:<16} | {'Low' if qname_template_diff_count == 0 else 'Medium':<8} | {'No action needed' if qname_template_diff_count == 0 else 'Review added/missing questions':<30} |\n"

if previousround_questionnaire.strip().lower() == "yes":
    qname_previous_diff_count = len(unique_previousround_questionnaire_qname)
    qname_previous_status = "✅ No Differences" if qname_previous_diff_count == 0 else "❌ Differences Found"
    qname_summary += f"| Previous Round     | {qname_previous_status:<19} | {qname_previous_diff_count:<16} | {'Low' if qname_previous_diff_count == 0 else 'Medium':<8} | {'No action needed' if qname_previous_diff_count == 0 else 'Check removed questions from new round':<30} |\n"

qname_country_diff_count = len(unique_country_questionnaire_qname)
qname_country_status = "✅ All Published" if qname_country_diff_count == 0 else "⚠️ Not Published"
qname_summary += f"| Current Quest. Only | {qname_country_status:<19} | {qname_country_diff_count:<16} | Low      | {'No action needed' if qname_country_diff_count == 0 else 'Validate if should be on the hub':<30} |\n"

execution_messages += qname_summary
print(qname_summary)

show_differences = False  # Set to False if you want to suppress extra detail
diff_threshold = 2       # Only show diffs if count > 2 when show_differences is False


# -- Detailed Differences Output (in revised order) --
if previousround_questionnaire.strip().lower() == "yes" and (show_differences or qname_previous_diff_count > diff_threshold):
    section = "\nQ Name in Previous Round but Missing in Current Round:\n"
    execution_messages += section
    print(section.strip())
    for q in sorted(unique_previousround_questionnaire_qname):
        execution_messages += f"- {q}\n"
        print(f"- {q}")

if show_differences or qname_template_diff_count > diff_threshold:
    section = "\nQ Name in Template but Missing in Current Questionnaire:\n"
    execution_messages += section
    print(section.strip())
    for q in sorted(unique_template_questionnaire_qname):
        execution_messages += f"- {q}\n"
        print(f"- {q}")

if show_differences or qname_country_diff_count > diff_threshold:
    section = "\nQ Name in Current Questionnaire not in Template (i.e., not published on the Hub):\n"
    execution_messages += section
    print(section.strip())
    for q in sorted(unique_country_questionnaire_qname):
        execution_messages += f"- {q}\n"
        print(f"- {q}")

# -- Optional Visual Diff --
if previousround_questionnaire.strip().lower() == "yes":
    geopoll_functions.highlight_differences_in_qname(
        country_questionnaire_file,
        previousround_questionnaire_file
    )

DETECTING DIFFERENCES IN FIELD: Q Name

Q Name Differences Summary
| Compared To       | Status             | Difference Count | Severity | Action                         |
|-------------------|---------------------|------------------|----------|--------------------------------|
| Template           | ❌ Differences Found | 72               | Medium   | Review added/missing questions |
| Current Quest. Only | ⚠️ Not Published    | 96               | Low      | Validate if should be on the hub |

Q Name in Template but Missing in Current Questionnaire:
- callback
- contact_2
- contact_2_name
- contact_3
- contact_3_name
- covid
- crp_all
- cs_crisis_consume_seed_stock
- cs_crisis_decrease_input_exp
- cs_crisis_harv_immature_crops
- cs_crisis_no_school
- cs_crisis_reduced_health_exp
- cs_crisis_sold_prod_assets
- cs_emergency_begged
- cs_emergency_hh_migration
- cs_emergency_illegal
- cs_emergency_sold_house
- cs_emergency_sold_last_female
- cs_stress_borrowed_money
- cs_stress_borrowed_o

The below are mandatory Rules that should be respected

In [ ]:
print("CHECKING HH_WEALTH_ QUESTION RULES\n")

wealth_questions = [q for q in C_qname_list_questions if q.startswith('hh_wealth_')]
other_wealth_questions = [q for q in C_qname_list_questions if q.startswith('o_hh_wealth_')]

if previousround_questionnaire.strip().lower() == "yes":
    previous_wealth_questions = [q for q in P_qname_list_questions if q.startswith('hh_wealth_')]
    previous_other_wealth = [q for q in P_qname_list_questions if q.startswith('o_hh_wealth_')]
else:
    previous_wealth_questions = []
    previous_other_wealth = []

if len(wealth_questions) >= 1:
    listed_wealth = ", ".join(sorted(wealth_questions))
    msg_1 = f"✔️ {len(wealth_questions)} `hh_wealth_` question(s) found: rule respected.\n   ➤ {listed_wealth}"
else:
    msg_1 = "❌ No `hh_wealth_` question found: at least one is required!"

if other_wealth_questions:
    listed_opt_wealth = ", ".join(sorted(other_wealth_questions))
    msg_2 = f"ℹ️ {len(other_wealth_questions)} optional `o_hh_wealth_` question(s) found.\n   ➤ {listed_opt_wealth}"
else:
    msg_2 = "ℹ️ No optional `o_hh_wealth_` questions found."

if previousround_questionnaire.strip().lower() == "yes":
    msg_3 = "\n📋 Wealth Questions in Previous Round:\n"
    if previous_wealth_questions:
        msg_3 += "   - Main: " + ", ".join(sorted(previous_wealth_questions)) + "\n"
    else:
        msg_3 += "   - Main: None found\n"

    if previous_other_wealth:
        msg_3 += "   - Optional: " + ", ".join(sorted(previous_other_wealth))
    else:
        msg_3 += "   - Optional: None found"
else:
    msg_3 = ""

print(msg_1)
print(msg_2)
if msg_3: print(msg_3)

execution_messages += f"\n{msg_1}\n{msg_2}"
if msg_3: execution_messages += f"\n{msg_3}"

CHECKING HH_WEALTH_ QUESTION RULES

✔️ 1 `hh_wealth_` question(s) found: rule respected.
   ➤ hh_wealth_water
ℹ️ 1 optional `o_hh_wealth_` question(s) found.
   ➤ o_hh_wealth_house


In [ ]:
print("\nCHECKING THAT RULES ON CS QUESTIONS ARE RESPECTED")

cs_questions = [q for q in C_qname_list_questions if q.startswith('cs')]
cs_stress = [q for q in cs_questions if q.startswith('cs_stress')]
cs_crisis = [q for q in cs_questions if q.startswith('cs_crisis')]
cs_emergency = [q for q in cs_questions if q.startswith('cs_emergency')]

total_cs = len(cs_questions)
num_stress = len(cs_stress)
num_crisis = len(cs_crisis)
num_emergency = len(cs_emergency)

if previousround_questionnaire.strip().lower() == "yes":
    prev_cs = [q for q in P_qname_list_questions if q.startswith('cs')]
    prev_stress = [q for q in prev_cs if q.startswith('cs_stress')]
    prev_crisis = [q for q in prev_cs if q.startswith('cs_crisis')]
    prev_emergency = [q for q in prev_cs if q.startswith('cs_emergency')]

    same_cs = (
        sorted(cs_stress) == sorted(prev_stress) and
        sorted(cs_crisis) == sorted(prev_crisis) and
        sorted(cs_emergency) == sorted(prev_emergency)
    )

    if same_cs:
        prev_summary = "\n✅ Same coping strategies appear in the previous round."
    else:
        prev_summary = (
            "\nDifferent CS questions appear in the previous round:\n📋 CS Questions in Previous Round:\n"
            f"- STRESS questions: {len(prev_stress)} ({', '.join(sorted(prev_stress)) or 'None'})\n"
            f"- CRISIS questions: {len(prev_crisis)} ({', '.join(sorted(prev_crisis)) or 'None'})\n"
            f"- EMERGENCY questions: {len(prev_emergency)} ({', '.join(sorted(prev_emergency)) or 'None'})"
        )
else:
    prev_summary = ""

if total_cs == 0:
    message_cs = (
        "\n❌ No CS questions found in the country questionnaire.\n"
        "Each respondent profile should receive at least 10 coping strategies.\n"
        "Please add at least 4 STRESS, 3 CRISIS, and 3 EMERGENCY questions."
    )
elif num_stress >= 4 and num_crisis >= 3 and num_emergency >= 3:
    message_cs = (
        f"\n✅ {total_cs} CS questions found in the country questionnaire.\n"
        "Rules respected — there are more than 10 coping strategies.\n"
        f"- STRESS questions: {num_stress} ({', '.join(sorted(cs_stress)) or 'None'})\n"
        f"- CRISIS questions: {num_crisis} ({', '.join(sorted(cs_crisis)) or 'None'})\n"
        f"- EMERGENCY questions: {num_emergency} ({', '.join(sorted(cs_emergency)) or 'None'})"
    )
else:
    message_cs = (
        f"\n⚠️ {total_cs} CS questions found in the country questionnaire.\n"
        "There are fewer than the required coping strategies.\n"
        "Minimum expected:\n"
        "- 4 STRESS questions\n"
        "- 3 CRISIS questions\n"
        "- 3 EMERGENCY questions\n"
        f"Current count:\n"
        f"- STRESS = {num_stress} ({', '.join(sorted(cs_stress)) or 'None'})\n"
        f"- CRISIS = {num_crisis} ({', '.join(sorted(cs_crisis)) or 'None'})\n"
        f"- EMERGENCY = {num_emergency} ({', '.join(sorted(cs_emergency)) or 'None'})"
    )

print(message_cs)
if prev_summary:
    print(prev_summary)

execution_messages += message_cs + "\n"
if prev_summary:
    execution_messages += prev_summary + "\n"


CHECKING THAT RULES ON CS QUESTIONS ARE RESPECTED

❌ No CS questions found in the country questionnaire.
Each respondent profile should receive at least 10 coping strategies.
Please add at least 4 STRESS, 3 CRISIS, and 3 EMERGENCY questions.


In [ ]:
print("\nCHECKING THAT RULES ON HDDS QUESTIONS ARE RESPECTED")

hdds_questions = [q for q in C_qname_list_questions if q.startswith('hdds')]
num_hdds = len(hdds_questions)

if previousround_questionnaire.strip().lower() == "yes":
    prev_hdds_questions = [q for q in P_qname_list_questions if q.startswith('hdds')]
    num_prev_hdds = len(prev_hdds_questions)

    same_hdds = sorted(hdds_questions) == sorted(prev_hdds_questions)

    if same_hdds:
        prev_summary = "\n✅ Same HDDS questions appear in the previous round."
    else:
        prev_summary = (
            "\nDifferent HDDS questions appear in the previous round:\n📋 HDDS Questions in Previous Round:\n"
            f"- Count: {num_prev_hdds}\n"
            f"- List: {', '.join(sorted(prev_hdds_questions)) or 'None'}"
        )
else:
    prev_summary = ""

if num_hdds == 0:
    message_hdds = (
        "\n❌ No HDDS questions found in the country questionnaire.\n"
        "You must include at least 3 HDDS questions."
    )
elif num_hdds < 2:
    message_hdds = (
        f"\n⚠️ Only {num_hdds} HDDS questions found in the country questionnaire.\n"
        "It is recommended to include more than 2 HDDS questions.\n"
        f"Found: {', '.join(sorted(hdds_questions)) or 'None'}"
    )
else:
    message_hdds = (
        f"\n✅ {num_hdds} HDDS questions found in the country questionnaire.\n"
        "Rule respected — more than 2 HDDS fields are present.\n"
        f"Found: {', '.join(sorted(hdds_questions)) or 'None'}"
    )

print(message_hdds)
if prev_summary:
    print(prev_summary)

execution_messages += message_hdds + "\n"
if prev_summary:
    execution_messages += prev_summary + "\n"


CHECKING THAT RULES ON HDDS QUESTIONS ARE RESPECTED

❌ No HDDS questions found in the country questionnaire.
You must include at least 3 HDDS questions.


In [ ]:
print("\nCHECKING THAT RULES ON CROP HARVEST QUESTIONS ARE RESPECTED")

expected_full = {
    "crp_harv_change",
    "crp_harv_vol",
    "crp_harv_unit",
    "crp_harv_unit_kg",
    "crp_harv_lastyr"
}

found = {q for q in C_qname_list_questions if q.startswith("crp_harv")}

if previousround_questionnaire.strip().lower() == "yes":
    prev_found = {q for q in P_qname_list_questions if q.startswith("crp_harv")}
    if sorted(found) == sorted(prev_found):
        prev_crp_summary = "\n✅ Same crop harvest questions appear in the previous round."
    else:
        prev_crp_summary = (
            "\nDifferent crop harvest questions appear in the previous round:\n📋 Crop Harvest Questions in Previous Round:\n"
            f"- Count: {len(prev_found)}\n"
            f"- List: {', '.join(sorted(prev_found)) or 'None'}"
        )
else:
    prev_crp_summary = ""

if found == {"crp_harv_change"}:
    message09 = (
        "\n✅ Only 'crp_harv_change' is included.\n"
        "Rule respected (minimal version).\n"
        f"Questions found: {', '.join(sorted(found))}"
    )

elif found == expected_full:
    message09 = (
        "\n✅ Full set of crop harvest questions is included.\n"
        "Rule respected.\n"
        f"Questions found: {', '.join(sorted(found))}"
    )

else:
    message09 = (
        "\n❌ Crop harvest question rule NOT respected.\n"
        "You must include EITHER:\n"
        "1. Only 'crp_harv_change', OR\n"
        "2. All 5 questions:\n"
        f"   {', '.join(sorted(expected_full))}\n"
        f"Questions found: {', '.join(sorted(found)) or 'None'}"
    )

print(message09)
if prev_crp_summary:
    print(prev_crp_summary)

execution_messages += message09 + "\n"
if prev_crp_summary:
    execution_messages += prev_crp_summary + "\n"


CHECKING THAT RULES ON CROP HARVEST QUESTIONS ARE RESPECTED

✅ Full set of crop harvest questions is included.
Rule respected.
Questions found: crp_harv_change, crp_harv_lastyr, crp_harv_unit, crp_harv_unit_kg, crp_harv_vol


In [ ]:
#Check the refused answers that are flagged but available in the current questionnaire !!!!! This is very important

import pandas as pd

print("COMPARISON OF DOMAIN TEXTS BETWEEN COUNTRY AND REFERENCE QUESTIONNAIRE\n")

def compare_domain_texts(reference_df, country_df, column_index, reference_name="Template"):
    for index, row in reference_df.iterrows():
        q_name = row.iloc[1]

        match_row = country_df[country_df.iloc[:, 1] == q_name]
        if not match_row.empty:
            ref_domain_lines = str(row.iloc[column_index]).split('\n')
            country_domain_lines = str(match_row.iloc[0, column_index]).split('\n')

            differences_found = False
            for i, ref_item in enumerate(ref_domain_lines, start=1):
                country_item = country_domain_lines[i - 1] if i < len(country_domain_lines) else "MISSING"
                if ref_item != country_item:
                    if not differences_found:
                        print(f"\n🧾 Q Name: {q_name}")
                        differences_found = True
                    print(f"{reference_name:<10} ➜ {ref_item}")
                    print(f"{'Country':<10} ➜ {country_item}")
            if differences_found:
                print("-" * 40)

country_df = pd.read_excel(country_questionnaire_file, header=None, skiprows=2)

if previousround_questionnaire.strip().lower() == "yes":
    reference_df = pd.read_excel(previousround_questionnaire_file, header=None, skiprows=2)
    reference_name = "Previous"
else:
    reference_df = pd.read_excel(template_questionnaire_file, header=None, skiprows=2)
    reference_name = "Template"

print(f"\n🔤 Comparing English Text (Column D) with {reference_name} Questionnaire")
compare_domain_texts(reference_df, country_df, column_index=3, reference_name=reference_name)

print(f"\n🌍 Comparing Local Language Text (Column F) with {reference_name} Questionnaire")
compare_domain_texts(reference_df, country_df, column_index=5, reference_name=reference_name)

print("\nDOMAIN COMPARISON COMPLETE.\n")

COMPARISON OF DOMAIN TEXTS BETWEEN COUNTRY AND REFERENCE QUESTIONNAIRE


🔤 Comparing English Text (Column D) with Template Questionnaire

🧾 Q Name: Q Name
Template   ➜ English
Country    ➜ MISSING
----------------------------------------

🧾 Q Name: Optin
Template   ➜ Optin
Country    ➜ MISSING
----------------------------------------

🧾 Q Name: calldispo
Template   ➜ 5)Non-Working Number
Country    ➜ MISSING
----------------------------------------

🧾 Q Name: NoAnswer
Template   ➜ Phone rang and no one answered. Inclusive of voicemail, answering machine and temporary issues with network connection. 
Country    ➜ MISSING
----------------------------------------

🧾 Q Name: NonWorkingNumber
Template   ➜ Phone number is not in service, not working, or not valid.
Country    ➜ MISSING
----------------------------------------

🧾 Q Name: resp_language
Template   ➜ 1)English
Country    ➜ 1)Burmese
Template   ➜ 2)other options [add as many as necessary]
Country    ➜ MISSING
---------------------

In [ ]:
#POSSIBLE ADDITION# Check the refused answers that are flagged but available in the current questionnaire
# !!!!! This is very important

import pandas as pd
import re

print("COMPARISON OF DOMAIN TEXTS BETWEEN COUNTRY AND REFERENCE QUESTIONNAIRE\n")

def compare_domain_texts(reference_df, country_df, column_index, reference_name="Template"):
    for index, row in reference_df.iterrows():
        q_name = row.iloc[1]

        match_row = country_df[country_df.iloc[:, 1] == q_name]
        if not match_row.empty:

            # 🔹 Clean and normalize domain text lines
            ref_domain_lines = [
                x.strip() for x in str(row.iloc[column_index]).split('\n') if x.strip()
            ]
            country_domain_lines = [
                x.strip() for x in str(match_row.iloc[0, column_index]).split('\n') if x.strip()
            ]

            differences_found = False

            for i, ref_item in enumerate(ref_domain_lines, start=1):
                country_item = (
                    country_domain_lines[i - 1]
                    if (i - 1) < len(country_domain_lines)
                    else "MISSING"
                )

                if ref_item != country_item:
                    if not differences_found:
                        print(f"\n🧾 Q Name: {q_name}")
                        differences_found = True

                    print(f"{reference_name:<10} ➜ {ref_item}")
                    print(f"{'Country':<10} ➜ {country_item}")

            if differences_found:
                print("-" * 40)


# 🔹 Load COUNTRY questionnaire
country_df = pd.read_excel(country_questionnaire_file, header=None, skiprows=2)

# 🔹 Load REFERENCE questionnaire (Previous or Template)
if previousround_questionnaire.strip().lower() == "yes":
    reference_df = pd.read_excel(previousround_questionnaire_file, header=None, skiprows=2)
    reference_name = "Previous"
else:
    reference_df = pd.read_excel(template_questionnaire_file, header=None, skiprows=2)
    reference_name = "Template"


print(f"\n🔤 Comparing English Text (Column D) with {reference_name} Questionnaire")
compare_domain_texts(reference_df, country_df, column_index=3, reference_name=reference_name)

print(f"\n🌍 Comparing Local Language Text (Column F) with {reference_name} Questionnaire")
compare_domain_texts(reference_df, country_df, column_index=5, reference_name=reference_name)

print("\nDOMAIN COMPARISON COMPLETE.\n")

COMPARISON OF DOMAIN TEXTS BETWEEN COUNTRY AND REFERENCE QUESTIONNAIRE


🔤 Comparing English Text (Column D) with Template Questionnaire

🧾 Q Name: NoAnswer
Template   ➜ Phone rang and no one answered. Inclusive of voicemail, answering machine and temporary issues with network connection.
Country    ➜ Phone rang and no one answered.
----------------------------------------

🧾 Q Name: NonWorkingNumber
Template   ➜ Phone number is not in service, not working, or not valid.
Country    ➜ Phone was disconnected or out of service.
----------------------------------------

🧾 Q Name: resp_language
Template   ➜ 1)English
Country    ➜ 1)Burmese
Template   ➜ 2)other options [add as many as necessary]
Country    ➜ 2)English [FOR TESTING ONLY]
----------------------------------------

🧾 Q Name: introduction
Template   ➜ Hello sir/ma'am, my name is #OPERATOR#, and I am calling from GeoPoll Polling Agency on behalf of the Food and Agriculture Organization of the United Nations [FAO]. We are conductin

In [ ]:
print("CREATE OUTPUT VALIDATED QUESTIONNAIRE FILE")

# Define the output path and filename
country_questionnaire_edited_file = os.path.join(
    temp_path,
    f"validated_questionnaire_{enumerator}_{lang}_{iso3}_{now}.xlsx"
)

# Copy the original country questionnaire to a new path
copyfile(country_questionnaire_file, country_questionnaire_edited_file)

# Construct and log the output message
message = f"\n\nOutput validated country questionnaire: {country_questionnaire_edited_file}"
print(message)
execution_messages += message

# Load the copied file into an Excel workbook
wb = openpyxl.load_workbook(country_questionnaire_edited_file)

# If previous round is available, generate the question changes sheet
if previousround_questionnaire.strip().lower() == "yes":
    from geopoll_functions import create_question_changes_sheet
    unique_country_questionnaire = list(set(C_sqname_list_questions) - set(T_sqname_list_questions))

    # Ensure required variables are defined
    try:
        wb = create_question_changes_sheet(
            wb,
            unique_country_questionnaire_qname,
            unique_previousround_questionnaire_qname,
            unique_country_questionnaire,
            unique_previousround_questionnaire
        )
    except NameError as e:
        print("Missing variables for change sheet generation:", e)
        execution_messages += "\nFailed to create question change sheet due to missing variables.\n"

# Save the workbook regardless
wb.save(country_questionnaire_edited_file)

CREATE OUTPUT VALIDATED QUESTIONNAIRE FILE


Output validated country questionnaire: C:\Users\edoar\WORK\FAO\temp\questionnaireValidationTest\validated_questionnaire_geopoll_en_MMR_20260225110038.xlsx


In [ ]:
# ---------------------------
# EDITING THE SURVEY BY REPLACING VALUES ACCORDING TO ADDITIONAL INFO SHEET
# ---------------------------

print("🛠️ EDITING THE SURVEY: Replacing values based on the 'Additional Info' sheet...")

# Ensure the edited questionnaire has been created and exists
assert os.path.exists(country_questionnaire_edited_file), "❌ Edited questionnaire file not found."

# If a previous round exists, use the template to update relevant content in the edited questionnaire
if previousround_questionnaire.strip().lower() == "yes":
    try:
        geopoll_modified.update_questionnaire(template_questionnaire_file, country_questionnaire_edited_file)
    except Exception as e:
        print(f"❌ Failed to update questionnaire with template values: {e}")
        execution_messages += "\n❌ Error during template value update.\n"

# Replace predefined string values in the edited questionnaire (e.g., based on "Additional Info" sheet)
try:
    geopoll_modified.find_and_replace_strings_in_df(country_questionnaire_edited_file)
    print("✅ Replacement from 'Additional Info' sheet completed.")
    execution_messages += "\n\n✅ Survey edited successfully based on the Additional Info sheet."
except Exception as e:
    print(f"❌ Error during find/replace operation: {e}")
    execution_messages += "\n❌ Error during find/replace from Additional Info sheet.\n"

🛠️ EDITING THE SURVEY: Replacing values based on the 'Additional Info' sheet...
❌ Error during find/replace operation: Worksheet named 'Additional information' not found


In [ ]:
# ---------------------------
# INSERT ADMIN SHEET WITH UPDATED BOUNDARIES
# ---------------------------

print("🗺️ INSERTING ADMIN SHEET WITH UPDATED BOUNDARIES...")

try:
    # Attempt to insert the admin reference sheet
    df = geopoll_modified.insert_sheet_with_adm_reference(country_questionnaire_edited_file, admin_level, iso3)
    
    # Show a preview of the inserted data (optional)
    display(df.head())  # Use 'display()' instead of 'print()' for Jupyter

    # Log success
    print("✅ Admin Info sheet inserted successfully.")
    execution_messages += "\n\n✅ Created Admin Info sheet with updated boundaries."

except Exception as e:
    print(f"❌ Failed to insert Admin Info sheet: {e}")
    execution_messages += f"\n\n❌ Error inserting Admin Info sheet: {e}"

#add a constrstraints to overwrite the existing one.

🗺️ INSERTING ADMIN SHEET WITH UPDATED BOUNDARIES...
❌ Failed to insert Admin Info sheet: Sheet 'Admin 2 info' already exists and if_sheet_exists is set to 'error'.


In [ ]:
# ---------------------------
# SORT CROP LIST BASED ON PRIORITIZATION
# ---------------------------

print("🌾 EDITING THE SURVEY BY SORTING THE CROP LIST VALUES ACCORDING TO PRIORITY...")

try:
    # Call the function to sort the crop list and retrieve the number of most common crops selected
    n_of_choices = geopoll_modified.sort_crop_list_by_selection(country_questionnaire_edited_file)

    # Build and print confirmation message
    message13 = f"\n✅ Number of most common crops selected in the Crop list sheet: {n_of_choices}"
    print(message13)

    # Append message to execution summary
    execution_messages += "\n\n✅ Survey edited by sorting the crop list according to prioritised crops." + message13

except Exception as e:
    # Handle any issues gracefully
    print(f"❌ Error while sorting crop list: {e}")
    execution_messages += f"\n\n❌ Error during crop list sorting: {e}"

🌾 EDITING THE SURVEY BY SORTING THE CROP LIST VALUES ACCORDING TO PRIORITY...

✅ Number of most common crops selected in the Crop list sheet: 10


In [ ]:
print("📝 CREATE OUTPUT FILE WITH EXECUTION MESSAGES")

try:
    # Construct the report file path
    text_file = os.path.join(temp_path, f"questionnaire_validation_report_{iso3}_R{round_number}_{now}.txt")

    # Write the execution messages to the file
    with open(text_file, "w", encoding="utf-8") as file:
        file.write(execution_messages)

    # Confirm completion
    print(f"✅ Validation report successfully saved:\n{text_file}")
    execution_messages += f"\n\n✅ Validation report saved to: {text_file}"

except Exception as e:
    print(f"❌ Failed to save validation report: {e}")
    execution_messages += f"\n\n❌ Error saving validation report: {e}"

📝 CREATE OUTPUT FILE WITH EXECUTION MESSAGES
✅ Validation report successfully saved:
C:\Users\edoar\WORK\FAO\temp\questionnaireValidationTest\questionnaire_validation_report_MMR_R13_20260225110038.txt


In [ ]:

print("⏱️ Execution time:", datetime.now() - startTime)
execution_messages += f"\n\n⏱️ Execution time: {datetime.now() - startTime}"

⏱️ Execution time: 0:01:35.018903
